# 06 - Launch and monitor the main grid
Long runs go through `scripts/run_sweep.py` in a terminal, never in this kernel. Each run resumes
from `checkpoints/<run_id>/last.pt` after an interruption, and finished runs are skipped.
Set `LAUNCH = True` to start a sweep in a new console window; otherwise this notebook only
reports progress and GPU-hours from the JSON logs.

In [1]:
# parameters
CONFIGS = ["configs/pilot_addition.yaml", "configs/tune_gates.yaml", "configs/main_addition.yaml",
           "configs/main_parity.yaml", "configs/main_gates.yaml", "configs/ablations.yaml"]
LAUNCH = False

In [2]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [3]:
import subprocess
from glwm2.eval.results import configs_from_yaml, finished
from glwm2.train.trainer import run_dirs
from glwm2.utils.io import read_json, write_json
status = []
for f in CONFIGS:
    cs = configs_from_yaml(PROJECT_ROOT / f)
    fin = finished(cs)
    secs = 0.0
    for c in fin:
        d = run_dirs(c)[1]
        secs += read_json(d / "train_summary.json")["train_seconds"] if (d / "train_summary.json").exists() else 0
        secs += read_json(d / "eval_summary.json")["eval_seconds"] if (d / "eval_summary.json").exists() else 0
    status.append(dict(config=f, runs=len(cs), finished=len(fin), gpu_hours=round(secs / 3600, 2)))
status = pd.DataFrame(status); status

,config,runs,finished,gpu_hours
0,configs/pilot_addition.yaml,8,8,3.47
1,configs/tune_gates.yaml,6,2,1.35
2,configs/main_addition.yaml,8,8,3.22
3,configs/main_parity.yaml,8,8,3.69
4,configs/main_gates.yaml,12,12,6.49
5,configs/ablations.yaml,3,3,0.83


In [4]:
if LAUNCH:
    todo = [f for f, r in zip(status.config, status.itertuples()) if r.finished < r.runs]
    cmd = [sys.executable, str(PROJECT_ROOT / "scripts" / "run_sweep.py"), *todo]
    subprocess.Popen(cmd, cwd=PROJECT_ROOT, creationflags=getattr(subprocess, "CREATE_NEW_CONSOLE", 0))
    print("launched:", " ".join(cmd))
else:
    print("to launch from a terminal:")
    print(r"  .venv\Scripts\python.exe scripts\run_sweep.py " + " ".join(CONFIGS))

to launch from a terminal:
  .venv\Scripts\python.exe scripts\run_sweep.py configs/pilot_addition.yaml configs/tune_gates.yaml configs/main_addition.yaml configs/main_parity.yaml configs/main_gates.yaml configs/ablations.yaml


In [5]:
from glwm2.utils.io import read_jsonl
logs = []
for f in CONFIGS:
    for r in read_jsonl(RESULTS_DIR / "sweeps" / f"{Path(f).stem}.jsonl"):
        logs.append(dict(config=Path(f).stem, run_id=r["run_id"], status=r["status"], minutes=round(r["sec"] / 60, 1)))
logs = pd.DataFrame(logs)
print(logs.status.value_counts() if len(logs) else "no sweep logs yet")
logs.tail(10)

status
ok                                                                                                                                                                                                                                                                                                                                                                                                                                                      37
error: AcceleratorError("CUDA error: out of memory\nSearch for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n")    23
error: TypeError('RNG state must be a torch.ByteTensor')                                                   

,config,run_id,status,minutes
53,main_gates,addition-v1gate-own-s1-79865170,ok,27.6
54,main_gates,parity-ponder-own-s1-246a04ec,ok,24.4
55,main_gates,addition-v1gate-own-s0-565b75e2,ok,23.6
56,ablations,addition-base-fixed_prior-s0-a56f86a9,"error: AcceleratorError(""CUDA error: out of me...",10.5
57,ablations,addition-base-fixed_prior-s0-c1df2b74,"error: AcceleratorError(""CUDA error: out of me...",0.0
58,ablations,addition-base-fixed_prior-s0-647caf4a,"error: AcceleratorError(""CUDA error: out of me...",0.0
59,ablations,addition-base-fixed_prior-s0-a56f86a9,error: TypeError('RNG state must be a torch.By...,0.0
60,ablations,addition-base-fixed_prior-s0-c1df2b74,ok,23.3
61,ablations,addition-base-fixed_prior-s0-647caf4a,ok,19.3
62,ablations,addition-base-fixed_prior-s0-a56f86a9,ok,7.3


In [6]:
WRITTEN.append(write_json(RESULTS_DIR / "grid_status.json", {"status": status.to_dict(orient="records"),
                                                             "total_gpu_hours": float(status.gpu_hours.sum())}))
print("total GPU-hours so far:", round(float(status.gpu_hours.sum()), 2))

total GPU-hours so far: 19.05


In [7]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))

Files written by this notebook:
  results\grid_status.json
